# 지저분한 데이터 청소하기

> 파이썬 3강 · 데이터 다루기

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [지저분한 데이터 청소하기](https://mioon1402.github.io/timeseriesdata/python/p03-cleaning.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.** 예시 데이터를 내려받습니다.

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales_messy.csv

# 표를 글자로 찍을 때 한글 열이 어긋나지 않게 (한글을 두 칸으로 계산)
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. 먼저 진단부터

**3-1. 첫인상 보기**

In [ ]:
import pandas as pd
messy = pd.read_csv("cafe_sales_messy.csv")

print(messy.head(6).to_string())
print()
print(messy.dtypes)
print("행 수:", len(messy))

**3-2. 정밀 진단**

In [ ]:
print("날짜 형식 예시:", messy["날짜"].head(4).tolist())
print("매출 표기 예시:", messy["매출"].head(4).tolist())
print()
print("요일 고유값:", sorted(messy["요일"].dropna().unique().tolist()))
print()
print("중복 행 개수:", messy.duplicated().sum())
print("결측 현황:")
messy.isna().sum()

## 2. 공백 제거

**3-3. strip 으로 앞뒤 공백 자르기**

In [ ]:
df = messy.copy()          # 원본은 남겨두고 사본에서 작업
df["요일"] = df["요일"].str.strip()

print("정리 후:", sorted(df["요일"].unique().tolist()))
print("고유값 개수:", df["요일"].nunique(), "개")

## 3. 중복 행 처리

**3-4. 중복 확인하고 지우기**

In [ ]:
# 어떤 행이 중복인지 먼저 눈으로 확인한다
중복행 = df[df.duplicated(keep=False)].sort_values("날짜")
print("중복 관련 행:", len(중복행), "개")
print(중복행.head(6).to_string())

# 확인했으면 제거
before = len(df)
df = df.drop_duplicates()
print(f"\n{before} → {len(df)}행 ({before - len(df)}개 제거)")

## 4. 문자열을 숫자로

**3-5. 매출을 진짜 숫자로**

In [ ]:
s = df["매출"].astype(str)                        # 먼저 전부 문자열로 통일
s = s.str.replace(",", "", regex=False)           # 천 단위 쉼표 제거
s = s.str.replace("원", "", regex=False)          # 단위 제거
s = s.str.strip()                                 # 남은 공백 제거
s = s.replace({"N/A": None, "-": None, "": None, "nan": None})  # 결측 표기 통일

df["매출"] = pd.to_numeric(s, errors="coerce")    # 숫자로 변환

print(df["매출"].head(6).tolist())
print("자료형:", df["매출"].dtype)
print("결측:", df["매출"].isna().sum(), "개")
print("평균:", f"{df['매출'].mean():,.0f}원")

## 5. 결측치 — 지울까 채울까

**3-6. 세 방법 비교해보기**

In [ ]:
원본 = df["매출"]

print(f"원본       n={원본.notna().sum()}  결측 {원본.isna().sum()}개")
print(f"행 삭제    n={len(원본.dropna())}")
print(f"평균 대체  n={len(원본.fillna(원본.mean()))}")
print(f"앞값 채움  n={len(원본.ffill())}")
print()
print("평균 대체값:", f"{원본.mean():,.0f}원")
print("중앙값 대체값:", f"{원본.median():,.0f}원")

## 6. 평균으로 채우면 무슨 일이 생기나

**3-7. 결측 대체 실험**

In [ ]:
import numpy as np
clean = pd.read_csv("cafe_sales.csv")
원본 = clean["sales"].dropna()

# 일부러 30%를 결측으로 만든다
rng = np.random.default_rng(42)
구멍 = 원본.copy()
구멍[rng.random(len(구멍)) < 0.3] = np.nan

삭제 = 구멍.dropna()
대체 = 구멍.fillna(구멍.mean())

print(f"진짜 원본   n={len(원본):3d}  평균 {원본.mean():>9,.0f}  표준편차 {원본.std():>9,.0f}")
print(f"행 삭제     n={len(삭제):3d}  평균 {삭제.mean():>9,.0f}  표준편차 {삭제.std():>9,.0f}")
print(f"평균 대체   n={len(대체):3d}  평균 {대체.mean():>9,.0f}  표준편차 {대체.std():>9,.0f}")
print()
print(f"→ 평균 대체를 하면 표준편차가 {(1 - 대체.std()/삭제.std())*100:.1f}% 줄어듭니다")

## 7. 청소 전체 흐름

**3-8. 청소 파이프라인**

In [ ]:
def 청소하기(경로):
    d = pd.read_csv(경로, na_values=["N/A", "-", ""])

    # 1) 문자열 공백 정리
    for col in d.select_dtypes(include=["object", "str"]).columns:
        d[col] = d[col].astype(str).str.strip()

    # 2) 중복 제거
    d = d.drop_duplicates()

    # 3) 숫자여야 할 열을 숫자로
    매출 = (d["매출"].astype(str)
                    .str.replace(",", "", regex=False)
                    .str.replace("원", "", regex=False)
                    .replace({"nan": None, "None": None}))
    d["매출"] = pd.to_numeric(매출, errors="coerce")

    return d

깨끗 = 청소하기("cafe_sales_messy.csv")
print("행 수:", len(깨끗))
print(깨끗.dtypes)
print()
깨끗.head(4)

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 청소한 데이터(깨끗)에서 평균기온의 결측이 몇 개인지 세어보세요.


# 문제 2. 요일별 평균 방문객수를 구해보세요.
#        힌트: 깨끗.groupby("요일")["방문객수"].mean()   ← 6강에서 자세히 배웁니다


# 문제 3. 매출 결측을 '중앙값'으로 채운 뒤, 표준편차가 얼마나 변하는지 비교해보세요.

**모범 답안**

In [ ]:
# 문제 1
print("평균기온 결측:", 깨끗["평균기온"].isna().sum(), "개")

# 문제 2
print()
print(깨끗.groupby("요일")["방문객수"].mean().round(1))

# 문제 3
원본 = 깨끗["매출"]
채움 = 원본.fillna(원본.median())
print()
print(f"삭제 후 표준편차 {원본.dropna().std():,.0f}")
print(f"중앙값 대체 후  {채움.std():,.0f}")

---

전체 강의 목록 → [눈으로 보는 통계](https://mioon1402.github.io/timeseriesdata/)